### **Validate configs**

In [9]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents)
    if (path / "pyproject.toml").exists()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Wrong location.")

load_dotenv(PROJECT_ROOT/".env", override=True)

required_variables = ["OPENAI_API_KEY",
"LLM_MODEL",
"AZURE_OPENAI_API_KEY",
"AZURE_OPENAI_BASE_URL",
"AZURE_OPENAI_DEPLOYMENT"]

missing = [
    name for name in required_variables
    if not os.getenv(name, "").strip()
]

if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

azure_base_url = os.environ["AZURE_OPENAI_BASE_URL"].strip().rstrip("/") + "/"

if not azure_base_url.endswith("/openai/v1/"):
    raise ValueError(
        "AZURE_OPENAI_BASE_URL must be the Azure OpenAI v1 URL "
        "ending in /openai/v1/."
    )

print("Configuration loaded. All required values are present.")

Configuration loaded. All required values are present.


In [12]:
openai_model_name = os.getenv("LLM_MODEL")

openai_llm = ChatOpenAI(
    model=openai_model_name,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
    max_tokens=128,
    timeout=60,
    max_retries=0,
)

azure_llm = ChatOpenAI(
    model=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=azure_base_url,
    reasoning_effort="minimal",
    max_completion_tokens=1024,
    timeout=60,
    max_retries=0,
)

print("Both clients created. No model calls made yet.")

Both clients created. No model calls made yet.


#### ***Connection Test***

In [13]:
responses = {}

def test_provider(provider_name, model):
    try:
        response = model.invoke(
            "Reply with exactly: Aegis connection successful."
        )
    except Exception as exc:
        # Avoid printing raw exception messages or request details.
        print(f"{provider_name}: FAILED")
        print(f"Error type: {type(exc).__name__}")

        status_code = getattr(exc, "status_code", None)
        if status_code is not None:
            print(f"HTTP status: {status_code}")

        return None

    responses[provider_name] = response

    print(f"Provider: {provider_name}")
    print(f"Response: {response.content}")
    print(f"Token usage: {response.usage_metadata}")
    print(f"Finish reason: {response.response_metadata.get('finish_reason')}")

    return response

In [14]:
openai_response = test_provider("openai", openai_llm)

Provider: openai
Response: Aegis connection successful.
Token usage: {'input_tokens': 16, 'output_tokens': 9, 'total_tokens': 25, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Finish reason: stop


In [15]:
azure_response = test_provider("azure", azure_llm)

Provider: azure
Response: Aegis connection successful.
Token usage: {'input_tokens': 16, 'output_tokens': 15, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Finish reason: stop


In [16]:
from aegis.config import get_settings
from aegis.llm.gateway import get_model
from aegis.llm.usage import extract_usage

settings = get_settings()
model = get_model("azure")

try:
    response = model.invoke("Reply with exactly: Aegis connection successful.")
except Exception as exc:
    print(f"Request failed: {type(exc).__name__}")
else:
    print(response.content)
    print(
        extract_usage(
            response,
            provider="azure",
            requested_model=settings.azure_openai_deployment,
        )
    )

Aegis connection successful.
{'provider': 'azure', 'requested_model': 'gpt-5-mini', 'reported_model': 'gpt-5-mini-2025-08-07', 'input_tokens': 16, 'output_tokens': 15, 'total_tokens': 31, 'cached_input_tokens': 0, 'reasoning_tokens': 0}


## Cost calculation

Convert token usage into an estimated USD token cost using the
configured price table.

Cached input is a subset of total input.
Reasoning tokens are already included in output tokens.

In [1]:
from decimal import Decimal

from aegis.llm.pricing import calculate_cost


# Synthetic usage: this cell does not call a provider.
usage = {
    "provider": "openai",
    "requested_model": "gpt-5.4",
    "input_tokens": 1000,
    "cached_input_tokens": 0,
    "output_tokens": 500,
    "reasoning_tokens": 200,
}

cost = calculate_cost(usage)

# Independently calculated using the configured rates:
# Input:  1000 × $2.50 / 1,000,000 = $0.0025
# Output:  500 × $15.00 / 1,000,000 = $0.0075
# Total:                             $0.0100
assert cost == Decimal("0.0100")

print("Estimated token cost, USD:", cost)
print("PASS: cost matches the manual calculation")

Estimated token cost, USD: 0.01
PASS: cost matches the manual calculation


In [2]:
# Now assume 500 of the 1,000 input tokens were served from cache.
cached_usage = {
    **usage,
    "cached_input_tokens": 500,
}

cached_cost = calculate_cost(cached_usage)

# Uncached input: 500 × $2.50 / 1M = $0.00125
# Cached input:   500 × $0.25 / 1M = $0.000125
# Output:         500 × $15.00 / 1M = $0.0075
# Total:                             $0.008875
assert cached_cost == Decimal("0.008875")
assert cached_cost < cost

print("Cost with cached input, USD:", cached_cost)
print("PASS: cached tokens use the lower configured rate")

Cost with cached input, USD: 0.008875
PASS: cached tokens use the lower configured rate
